# Employee Retention & Demographic Analysis using PySpark

**Objective:** Perform scalable Exploratory Data Analysis (EDA) on HR records using PySpark DataFrames. This project focuses on data manipulation, filtering, and aggregation to uncover actionable trends in employee retention, experience, and demographics.

**Key Techniques:** Schema management, distributed data filtering, and statistical aggregations.

In [1]:
import findspark

In [2]:
findspark.init()

In [3]:
import pyspark

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.getOrCreate()

In [6]:
spark

In [7]:
from pyspark.sql.functions import col, avg

df = spark.read.csv("C:/Users/user/Downloads/data.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()

+---------+-----------+---------+-----------+---+------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-------------------------+----------+
|Bachelors|       2017|Bangalore|          3| 34|  Male|                        0|         0|
|Bachelors|       2013|     Pune|          1| 28|Female|                        3|         1|
|Bachelors|       2014|New Delhi|          3| 38|Female|                        2|         0|
|  Masters|       2016|Bangalore|          3| 27|  Male|                        5|         1|
|  Masters|       2017|     Pune|          3| 24|  Male|                        2|         1|
+---------+-----------+---------+-----------+---+------+-------------------------+----------+
only showing top 5 rows
root
 |-- Education: string (nullable = true)
 |-- JoiningYear: integer (nullable = true)
 |-- City: string (nullable = true)
 |--

### Count the number of employees in each city.

In [8]:
df.groupBy("City").count().show()

+---------+-----+
|     City|count|
+---------+-----+
|Bangalore|   73|
|     Pune|   37|
|New Delhi|   40|
+---------+-----+



### Calculate the average ExperienceInCurrentDomain for each Education level.

In [9]:
df.groupBy("Education").agg(avg("ExperienceInCurrentDomain").alias("Avg_Experience")).show()

+---------+-----------------+
|Education|   Avg_Experience|
+---------+-----------------+
|  Masters|2.914285714285714|
|Bachelors|2.452830188679245|
|      PHD|3.111111111111111|
+---------+-----------------+



### Calculate the average age of employees in each city.

In [10]:
df.groupBy("City").agg(avg("Age").alias("Avg_Age")).show()

+---------+------------------+
|     City|           Avg_Age|
+---------+------------------+
|Bangalore|29.383561643835616|
|     Pune| 29.10810810810811|
|New Delhi|             29.15|
+---------+------------------+



### How many female employees have worked 2 years or more in the company?

In [11]:
female_2plus_exp = df.filter(
    (col("Gender") == "Female") & 
    (col("ExperienceInCurrentDomain") >= 2)
).count()

print(f"Female employees with 2+ years of experience: {female_2plus_exp}")

Female employees with 2+ years of experience: 38


### How many employees who are more than 27 years old have left the company?

In [12]:
left_older_27 = df.filter(
    (col("Age") > 27) & 
    (col("LeaveOrNot") == 1)
).count()

print(f"Employees older than 27 who left: {left_older_27}")

Employees older than 27 who left: 20


### Calculate the average age of employees who have more than 2 years of experience.

In [13]:
df.filter(col("ExperienceInCurrentDomain") > 2).agg(avg("Age").alias("Avg_Age_Exp_gt_2")).show()

+------------------+
|  Avg_Age_Exp_gt_2|
+------------------+
|28.253164556962027|
+------------------+



### How many male employees who have more than 2 years of experience stayed in the company?

In [14]:
male_stayed_exp = df.filter(
    (col("Gender") == "Male") & 
    (col("ExperienceInCurrentDomain") > 2) & 
    (col("LeaveOrNot") == 0)
).count()

print(f"Male employees with > 2 years experience who stayed: {male_stayed_exp}")

Male employees with > 2 years experience who stayed: 37
